# 21 — 29042675 Label Semantics Confirmation

## Objetivo

Confirmar la semántica de las etiquetas del dataset `29042675` antes de construir `H1_violation`, `H4_violation`, `H5_violation` y `H6_violation`.

Este notebook:

- analiza `README.rtf` y `EvaluationTemplate.rtf`;
- identifica la estructura real de los bloques del `IRR_Results.csv`;
- recupera evidencia directamente de los 43 documentos DOCX de evaluaciones LLM;
- intenta enlazar cada DOCX con `Prototype_Metadata.csv` mediante la URL del prototipo, no por nombre;
- genera **candidatos de etiqueta LLM_DIRECT** solo cuando la evidencia textual lo permite;
- no construye todavía el dataset maestro ni mezcla estas etiquetas con X de otras fuentes.


## 1. Configuración

In [1]:
from pathlib import Path
from urllib.parse import urlparse, unquote
from collections import defaultdict
import pandas as pd
import numpy as np
import re
import zipfile
import xml.etree.ElementTree as ET
import html

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 400)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

VT_DIR = (
    PUBLIC_RAW
    / "03_29042675"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("VT_DIR:", VT_DIR)


VT_DIR: /Users/uriel/Documents/machine_learning/data/public_v2/raw/03_29042675


## 2. Cargar metadata e IRR

In [2]:
metadata = pd.read_csv(
    VT_DIR
    / "Prototype_Metadata.csv"
)

irr = pd.read_csv(
    VT_DIR
    / "IRR_Results.csv"
)

print("Metadata:", metadata.shape)
print("IRR:", irr.shape)

display(metadata)


Metadata: (20, 4)
IRR: (104, 36)


,Prototype,Prototype Type,Domain,Scource URL
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0


## 3. Corregir la modalidad de las fuentes

En el notebook 20 una URL que no era Figma/Behance quedó provisionalmente como `OTHER_SOURCE`. Aquí no se interpreta automáticamente como fuente web. Una URL SharePoint de imagen/documento sigue siendo un artefacto, no HTML/CSS/JS de la interfaz evaluada.


In [3]:
def source_domain(value):

    if pd.isna(value):
        return None

    try:
        return (
            urlparse(str(value))
            .netloc
            .lower()
            .replace("www.", "")
        )
    except Exception:
        return None


def classify_source_modality(value):

    domain = source_domain(value)

    if domain is None:
        return "MISSING"

    if "figma.com" in domain:
        return "FIGMA_PROTOTYPE"

    if "behance.net" in domain:
        return "BEHANCE_PRESENTATION"

    if "sharepoint.com" in domain:
        return "SHAREPOINT_ARTIFACT"

    return "OTHER_ARTIFACT"


metadata_source_audit = metadata.copy()

metadata_source_audit[
    "source_domain"
] = metadata_source_audit[
    "Scource URL"
].apply(source_domain)

metadata_source_audit[
    "source_modality"
] = metadata_source_audit[
    "Scource URL"
].apply(classify_source_modality)

metadata_source_audit[
    "canonical_web_X_ready"
] = False

print(
    metadata_source_audit[
        "source_modality"
    ].value_counts(dropna=False)
)

display(metadata_source_audit)


source_modality
BEHANCE_PRESENTATION    11
FIGMA_PROTOTYPE          8
SHAREPOINT_ARTIFACT      1
Name: count, dtype: int64


,Prototype,Prototype Type,Domain,Scource URL,source_domain,source_modality,canonical_web_X_ready
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0,figma.com,FIGMA_PROTOTYPE,False
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?,figma.com,FIGMA_PROTOTYPE,False
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063,figma.com,FIGMA_PROTOTYPE,False
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,behance.net,BEHANCE_PRESENTATION,False
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441,figma.com,FIGMA_PROTOTYPE,False
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,behance.net,BEHANCE_PRESENTATION,False
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450,figma.com,FIGMA_PROTOTYPE,False
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521,figma.com,FIGMA_PROTOTYPE,False
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925,figma.com,FIGMA_PROTOTYPE,False
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,figma.com,FIGMA_PROTOTYPE,False


## 4. Extraer texto de DOCX

In [4]:
def extract_docx_text(path):

    try:
        with zipfile.ZipFile(path) as z:
            xml = z.read("word/document.xml")

        root = ET.fromstring(xml)

        chunks = []

        for element in root.iter():

            if (
                element.tag.endswith("}t")
                and
                element.text
            ):
                chunks.append(element.text)

        return " ".join(chunks)

    except Exception as e:

        print("DOCX error:", path.name, e)
        return ""


## 5. Extraer texto básico de RTF

In [5]:
def rtf_to_text(raw_text):

    # Caracteres hexadecimales RTF: \'hh
    def hex_replace(match):
        try:
            return bytes.fromhex(
                match.group(1)
            ).decode(
                "cp1252",
                errors="ignore"
            )
        except Exception:
            return ""

    text = re.sub(
        r"\\'([0-9a-fA-F]{2})",
        hex_replace,
        raw_text
    )

    # Unicode RTF: \uN?
    def unicode_replace(match):
        try:
            value = int(match.group(1))
            if value < 0:
                value += 65536
            return chr(value)
        except Exception:
            return ""

    text = re.sub(
        r"\\u(-?\d+)\??",
        unicode_replace,
        text
    )

    # Separadores comunes
    text = re.sub(
        r"\\par(?:d)?\b",
        "\n",
        text
    )

    text = re.sub(
        r"\\tab\b",
        "\t",
        text
    )

    # Eliminar destinos RTF y control words
    text = re.sub(
        r"\{\\\*[^{}]*\}",
        " ",
        text
    )

    text = re.sub(
        r"\\[a-zA-Z]+-?\d* ?",
        " ",
        text
    )

    text = text.replace("{", " ").replace("}", " ")

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n\s*\n+",
        "\n",
        text
    )

    return html.unescape(
        text
    ).strip()


rtf_texts = {}

for filename in [
    "README.rtf",
    "EvaluationTemplate.rtf"
]:

    path = VT_DIR / filename

    if not path.exists():
        print(filename, "❌")
        continue

    raw = path.read_text(
        encoding="latin-1",
        errors="ignore"
    )

    rtf_texts[
        filename
    ] = rtf_to_text(raw)

    print(
        "\n",
        "=" * 100
    )

    print(filename)

    print(
        rtf_texts[
            filename
        ][:6000]
    )



README.rtf
Title of Dataset: LLM-Based Heuristic Evaluations of High- and Low-Fidelity Prototypes (GPT-4o) 
 Author(s): Sehrish Basir Nizamani 
 Categories: Human-computer interaction, Artificial intelligence not elsewhere classified 
 Group: Computer Science 
 Item Type: dataset 
 Keywords: Usability Evaluation, Low- and High-Fidelity Prototypes, Large Language Models (LLMs), ChatGPT, Nielsenâs Heuristics, Heuristic evaluations, Prompt Engineering 
 Description: This dataset contains structured usability evaluation data for a range of user interface prototypes, assessed using Nielsen’s 10 heuristic principles. It includes:Prototype_Metadata.csv – Metadata for each prototype, specifying its name, fidelity level (high or low), domain, and source URL.IRR_Results.csv – Inter-Rater Reliability (IRR) results comparing human evaluator agreement across ten heuristics.EvaluationTemplate.rtf – A standardized evaluation template used by both human raters and GPT-4o to ensure consistent heuris

## 6. Buscar evidencia semántica en los RTF

In [6]:
RTF_TERMS = [
    "compare",
    "comparison",
    "inter-rater",
    "agreement",
    "overall rating",
    "severity",
    "0 to 4",
    "0-4",
    "evaluator",
    "evaluation 1",
    "evaluation 2",
    "chatgpt",
    "gemini",
    "heuristic"
]

rtf_evidence_records = []

for filename, text in rtf_texts.items():

    lower = text.lower()

    for term in RTF_TERMS:

        start_search = 0

        while True:

            idx = lower.find(
                term.lower(),
                start_search
            )

            if idx == -1:
                break

            start = max(
                0,
                idx - 300
            )

            end = min(
                len(text),
                idx + 700
            )

            rtf_evidence_records.append({
                "file":
                    filename,

                "term":
                    term,

                "context":
                    text[
                        start:end
                    ]
            })

            start_search = idx + len(term)

rtf_evidence = pd.DataFrame(
    rtf_evidence_records,
    columns=[
        "file",
        "term",
        "context"
    ]
)

print(
    "RTF evidence rows:",
    len(rtf_evidence)
)

display(
    rtf_evidence
)


RTF evidence rows: 50


,file,term,context
0,README.rtf,comparison,"wo phases (LLM Evaluation 1 and LLM Evaluation 2), each divided into high-fidelity and low-fidelity prototypes. Each prototype file contains issue identification, severity ratings (0–4), and heuristic-based recommendations.This dataset supports research on AI-assisted usability evaluation, enabling comparison between LLM and human assessments, and analysis of prototype usability across domains..."
1,README.rtf,comparison,"ataset and Description of Files \nPrototype_Metadata.csv – Contains metadata details for various prototypes, including attributes such as Prototype, Prototype Fidelity, Domain and Source URL \nIRR_Results.csv – This file contains inter-rater reliability (IRR) data for prototype evaluations, including comparison scores across ten heuristics (H1–H10) for multiple designs, along with links to pro..."
2,README.rtf,inter-rater,"set contains structured usability evaluation data for a range of user interface prototypes, assessed using Nielsen’s 10 heuristic principles. It includes:Prototype_Metadata.csv – Metadata for each prototype, specifying its name, fidelity level (high or low), domain, and source URL.IRR_Results.csv – Inter-Rater Reliability (IRR) results comparing human evaluator agreement across ten heuristics...."
3,README.rtf,inter-rater,"sponding Author E-mail Address: sehrishbasir@vt.edu \n Files/Folders in Dataset and Description of Files \nPrototype_Metadata.csv – Contains metadata details for various prototypes, including attributes such as Prototype, Prototype Fidelity, Domain and Source URL \nIRR_Results.csv – This file contains inter-rater reliability (IRR) data for prototype evaluations, including comparison scores acr..."
4,README.rtf,agreement,"user interface prototypes, assessed using Nielsen’s 10 heuristic principles. It includes:Prototype_Metadata.csv – Metadata for each prototype, specifying its name, fidelity level (high or low), domain, and source URL.IRR_Results.csv – Inter-Rater Reliability (IRR) results comparing human evaluator agreement across ten heuristics.EvaluationTemplate.rtf – A standardized evaluation template used..."
5,README.rtf,agreement,"uding attributes such as Prototype, Prototype Fidelity, Domain and Source URL \nIRR_Results.csv – This file contains inter-rater reliability (IRR) data for prototype evaluations, including comparison scores across ten heuristics (H1–H10) for multiple designs, along with links to prototypes and rater agreement metrics. \nEvaluationTemplate.rtf – A structured template used for heuristic evaluati..."
6,README.rtf,severity,"n raters and GPT-4o to ensure consistent heuristic assessment.LLM_Evaluation folder – Structured evaluations conducted by GPT-4o across two phases (LLM Evaluation 1 and LLM Evaluation 2), each divided into high-fidelity and low-fidelity prototypes. Each prototype file contains issue identification, severity ratings (0–4), and heuristic-based recommendations.This dataset supports research on AI..."
7,README.rtf,evaluator,"a range of user interface prototypes, assessed using Nielsen’s 10 heuristic principles. It includes:Prototype_Metadata.csv – Metadata for each prototype, specifying its name, fidelity level (high or low), domain, and source URL.IRR_Results.csv – Inter-Rater Reliability (IRR) results comparing human evaluator agreement across ten heuristics.EvaluationTemplate.rtf – A standardized evaluation tem..."
8,README.rtf,evaluator,"-rater reliability (IRR) data for prototype evaluations, including comparison scores across ten heuristics (H1–H10) for multiple designs, along with links to prototypes and rater agreement metrics. \nEvaluationTemplate.rtf – A structured template used for heuristic evaluations, guiding LLMs or human evaluators to assess prototypes based on Nielsen’s usability principles. \nLLM_Evaluation – Con..."
9,README.rtf,evaluation 1,"IRR) results comparing human evaluator agreement across ten heuristics.EvaluationTemplate.rtf – A standardized evaluation tem

## 7. Verificar matemáticamente la estructura de `H* Compare`

In [7]:
COMPARE_COLS = [
    "H1 Compare",
    "H4 Compare",
    "H5 Compare",
    "H6 Compare"
]

OVERALL_COLS = {
    "H1":
        "H1. Overal Rating (4)",

    "H4":
        "H4 Overall Rating",

    "H5":
        "H5 Overall Rating",

    "H6":
        "H6 Overall Rating"
}

compare_numeric = irr[
    COMPARE_COLS
].apply(
    pd.to_numeric,
    errors="coerce"
)

overall_numeric = irr[
    list(
        OVERALL_COLS.values()
    )
].apply(
    pd.to_numeric,
    errors="coerce"
)

print(
    "Compare non-missing por columna:"
)

print(
    compare_numeric.notna().sum()
)

print(
    "\nOverall non-missing por columna:"
)

print(
    overall_numeric.notna().sum()
)


Compare non-missing por columna:
H1 Compare    58
H4 Compare    58
H5 Compare    58
H6 Compare    58
dtype: int64

Overall non-missing por columna:
H1. Overal Rating (4)    40
H4 Overall Rating        40
H5 Overall Rating        40
H6 Overall Rating        40
dtype: int64


In [8]:
# Identificar filas donde Compare tiene únicamente valores binarios 0/1.
compare_binary_mask = (
    compare_numeric
    .apply(
        lambda row:
            row.dropna()
            .isin([0, 1])
            .all()
            if row.notna().any()
            else False,
        axis=1
    )
)

compare_summary_mask = (
    compare_numeric.notna().any(axis=1)
    &
    ~compare_binary_mask
)

print(
    "Filas Compare binarias:",
    int(
        compare_binary_mask.sum()
    )
)

print(
    "Filas Compare resumen/no binarias:",
    int(
        compare_summary_mask.sum()
    )
)

display(
    irr.loc[
        compare_summary_mask,
        [
            "Name",
            "Prototype Link"
        ]
        +
        COMPARE_COLS
    ]
)


Filas Compare binarias: 50
Filas Compare resumen/no binarias: 8


,Name,Prototype Link,H1 Compare,H4 Compare,H5 Compare,H6 Compare
11,NaN,NaN,10.0,10.0,10.0,10.0
12,NaN,NaN,9.0,7.0,7.0,6.0
24,NaN,NaN,10.0,10.0,10.0,10.0
25,NaN,NaN,9.0,7.0,9.0,10.0
77,NaN,NaN,10.0,10.0,10.0,10.0
78,NaN,NaN,8.0,7.0,7.0,6.0
92,NaN,NaN,10.0,10.0,10.0,10.0
93,NaN,NaN,5.0,6.0,9.0,4.0


In [9]:
# Verificar si algunas filas resumen son exactamente sumas de bloques binarios.
summary_rows = (
    irr.loc[
        compare_summary_mask,
        COMPARE_COLS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

binary_rows = (
    irr.loc[
        compare_binary_mask,
        [
            "Name"
        ]
        +
        COMPARE_COLS
    ]
    .copy()
)

for col in COMPARE_COLS:
    binary_rows[
        col
    ] = pd.to_numeric(
        binary_rows[
            col
        ],
        errors="coerce"
    )

print(
    "Suma total de las filas binarias:"
)

print(
    binary_rows[
        COMPARE_COLS
    ].sum()
)

print(
    "\nFilas resumen:"
)

display(
    summary_rows
)


Suma total de las filas binarias:
H1 Compare    40.0
H4 Compare    34.0
H5 Compare    39.0
H6 Compare    34.0
dtype: float64

Filas resumen:


,H1 Compare,H4 Compare,H5 Compare,H6 Compare
11,10.0,10.0,10.0,10.0
12,9.0,7.0,7.0,6.0
24,10.0,10.0,10.0,10.0
25,9.0,7.0,9.0,10.0
77,10.0,10.0,10.0,10.0
78,8.0,7.0,7.0,6.0
92,10.0,10.0,10.0,10.0
93,5.0,6.0,9.0,4.0


## 8. Inventario de evaluaciones LLM

La unidad de evidencia será un archivo DOCX de evaluación. No se asumirá que el nombre del archivo equivale al nombre del prototipo.


In [10]:
docx_files = sorted(
    [
        path
        for path
        in (
            VT_DIR
            / "LLM_Evaluations"
        ).rglob("*.docx")
        if path.is_file()
    ]
)

print(
    "DOCX encontrados:",
    len(docx_files)
)


DOCX encontrados: 43


In [11]:
def infer_evaluation_round(path):

    text = str(path)

    match = re.search(
        r"LLM Evaluation\s*(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(
            match.group(1)
        )

    return None


def infer_fidelity(path):

    text = str(path).lower()

    if "high fidelity" in text:
        return "HIGH_FIDELITY"

    if "low fidelity" in text:
        return "LOW_FIDELITY"

    return None


## 9. Normalizar URLs para enlazar DOCX ↔ metadata

In [12]:
URL_RE = re.compile(
    r"https?://[^\s<>\[\]\(\){}\"']+",
    flags=re.IGNORECASE
)


def clean_url_token(value):

    if value is None:
        return None

    value = (
        str(value)
        .strip()
        .replace("\u200b", "")
        .replace("\u202f", "")
        .replace("\xa0", "")
    )

    value = value.rstrip(
        ".,;:!?)]}>"
    )

    return value


def normalize_source_url(value):

    if pd.isna(value):
        return None

    value = clean_url_token(value)

    try:

        parsed = urlparse(value)

        host = (
            parsed.netloc
            .lower()
            .replace(
                "www.",
                ""
            )
        )

        path = unquote(
            parsed.path
        ).rstrip("/")

        return (
            host
            +
            path.lower()
        )

    except Exception:
        return None


metadata_url_registry = (
    metadata[
        [
            "Prototype",
            "Prototype Type",
            "Domain",
            "Scource URL"
        ]
    ]
    .copy()
)

metadata_url_registry[
    "normalized_url"
] = metadata_url_registry[
    "Scource URL"
].apply(
    normalize_source_url
)

display(
    metadata_url_registry
)


,Prototype,Prototype Type,Domain,Scource URL,normalized_url
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0,figma.com/design/oiozznzhizniealb5dpkmu/food-order-app--community-
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?,figma.com/design/1wjtuxkm0ofriwjzkd61n2/gaia-desktop-design-(community)
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063,figma.com/community/file/1271304360859402063
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,behance.net/gallery/198020739/time-tracker-mobile-application
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441,figma.com/community/file/1379689864793179441
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,behance.net/gallery/158363369/ui-projects/modules/893495869
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450,figma.com/community/file/988854619809579450
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521,figma.com/community/file/1250314760992299521
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925,figma.com/community/file/12725823068122925
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,figma.com/design/qlczcypaov1fshdefnx6t4/real-estate-agency---landing-page-design-(community)


In [13]:
metadata_url_map = defaultdict(list)

for _, row in metadata_url_registry.iterrows():

    key = row[
        "normalized_url"
    ]

    if key is None:
        continue

    metadata_url_map[
        key
    ].append(
        row[
            "Prototype"
        ]
    )


## 10. Construir registro de DOCX y enlazar por URL

In [14]:
doc_registry_records = []

doc_text_cache = {}

for path in docx_files:

    text = extract_docx_text(
        path
    )

    doc_text_cache[
        str(path)
    ] = text

    urls = [
        clean_url_token(x)
        for x in URL_RE.findall(
            text
        )
    ]

    normalized_urls = []

    for url in urls:

        normalized = normalize_source_url(
            url
        )

        if normalized:
            normalized_urls.append(
                normalized
            )

    matched_prototypes = []

    for normalized in normalized_urls:

        matched_prototypes.extend(
            metadata_url_map.get(
                normalized,
                []
            )
        )

    matched_prototypes = sorted(
        set(
            matched_prototypes
        )
    )

    if len(matched_prototypes) == 1:

        match_status = "MATCHED_BY_URL"
        selected_prototype = matched_prototypes[0]

    elif len(matched_prototypes) > 1:

        match_status = "AMBIGUOUS_URL_MATCH"
        selected_prototype = None

    else:

        match_status = "NO_URL_MATCH"
        selected_prototype = None

    doc_registry_records.append({
        "file":
            str(
                path.relative_to(
                    VT_DIR
                )
            ),

        "document_name":
            path.stem,

        "evaluation_round":
            infer_evaluation_round(
                path
            ),

        "fidelity":
            infer_fidelity(
                path
            ),

        "urls_found":
            len(
                normalized_urls
            ),

        "normalized_urls":
            " | ".join(
                sorted(
                    set(
                        normalized_urls
                    )
                )
            ),

        "match_status":
            match_status,

        "prototype":
            selected_prototype
    })

doc_registry = pd.DataFrame(
    doc_registry_records
)

print(
    doc_registry[
        "match_status"
    ].value_counts(
        dropna=False
    )
)

display(
    doc_registry
)


DOCX error: Bakery.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: E tutor.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Food Order.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: GAIA community app.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Hospital App 1.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Hospital App 2.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Housing.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Running app.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Artwork website.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Beer Garden.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: Blog App.docx "There is no item named 'word/document.xml' in the archive"
DOCX error: E-commerc

,file,document_name,evaluation_round,fidelity,urls_found,normalized_urls,match_status,prototype
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,Baking shop,1,HIGH_FIDELITY,1,figma.com/community/file/988854619809579450,MATCHED_BY_URL,Bakery
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,1,HIGH_FIDELITY,1,figma.com/community/file/1271304360859402063,MATCHED_BY_URL,E-Tutor
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,Food delivery,1,HIGH_FIDELITY,1,figma.com/design/oiozznzhizniealb5dpkmu/food-order-app-,NO_URL_MATCH,NaN
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Gaia.docx,Gaia,1,HIGH_FIDELITY,1,figma.com/design/1wjtuxkm0ofriwjzkd61n2/gaia-desktop-design-,NO_URL_MATCH,NaN
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,HospitalApp,1,HIGH_FIDELITY,1,figma.com/community/file/1250314760992299521,MATCHED_BY_URL,Hospital 1
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,HospitalApp2,1,HIGH_FIDELITY,1,figma.com/community/file/12725823068122925,MATCHED_BY_URL,Hospital 2
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Housing.docx,Housing,1,HIGH_FIDELITY,1,figma.com/design/qlczcypaov1fshdefnx6t4/real-estate-agency---landing-page-design-,NO_URL_MATCH,NaN
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Notification.docx,Notification,1,HIGH_FIDELITY,1,behance.net/gallery/198020739/time-tracker-mobile-application,MATCHED_BY_URL,Tracking App
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Running.docx,Running,1,HIGH_FIDELITY,1,behance.net/gallery/158363369/ui-projects/modules/893495869,MATCHED_BY_URL,Running
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Travel.docx,Travel,1,HIGH_FIDELITY,1,figma.com/community/file/1379689864793179441,MATCHED_BY_URL,Travel


## 11. Extraer secciones de H1, H4, H5 y H6

Se usan los nombres completos de Nielsen como delimitadores. La extracción es evidencia forense del documento; no es X del modelo.


In [15]:
HEURISTICS = {
    "H1":
        "Visibility of System Status",

    "H4":
        "Consistency and Standards",

    "H5":
        "Error Prevention",

    "H6":
        "Recognition Rather Than Recall"
}

ALL_HEURISTIC_HEADINGS = [
    "Visibility of System Status",
    "Match Between the System and the Real World",
    "User Control and Freedom",
    "Consistency and Standards",
    "Error Prevention",
    "Recognition Rather Than Recall",
    "Flexibility and Efficiency of Use",
    "Aesthetic and Minimalist Design",
    "Help Users Recognize, Diagnose, and Recover from Errors",
    "Help and Documentation"
]


def normalize_spaces(text):

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def extract_heuristic_section(
    text,
    heading
):

    normalized = normalize_spaces(
        text
    )

    start_match = re.search(
        re.escape(
            heading
        ),
        normalized,
        flags=re.IGNORECASE
    )

    if not start_match:
        return None

    start = start_match.start()

    candidate_ends = []

    for other in ALL_HEURISTIC_HEADINGS:

        if other.lower() == heading.lower():
            continue

        match = re.search(
            re.escape(
                other
            ),
            normalized[
                start_match.end():
            ],
            flags=re.IGNORECASE
        )

        if match:

            candidate_ends.append(
                start_match.end()
                +
                match.start()
            )

    end = (
        min(
            candidate_ends
        )
        if candidate_ends
        else len(
            normalized
        )
    )

    return normalized[
        start:end
    ]


## 12. Detectar evidencia de severidad y ausencia explícita de problemas

In [16]:
SEVERITY_PATTERNS = [
    re.compile(
        r"Severity\s*(?:Rating)?\s*[:\-]?\s*([0-4])(?:\.0)?\b",
        flags=re.IGNORECASE
    ),
    re.compile(
        r"Severity\s*[:\-]?\s*([0-4])\s*\(",
        flags=re.IGNORECASE
    )
]

OVERALL_PATTERNS = [
    re.compile(
        r"Overall\s+(?:Severity\s+)?Rating\s*[:\-]\s*([0-4])(?:\.0)?\b",
        flags=re.IGNORECASE
    ),
    re.compile(
        r"Overall\s+Rating\s*(?:is)?\s*[:\-]?\s*([0-4])(?:\.0)?\b",
        flags=re.IGNORECASE
    )
]

NO_ISSUE_PATTERNS = [
    re.compile(
        r"no\s+(?:major\s+)?(?:usability\s+)?issues?\s+(?:were\s+)?found",
        flags=re.IGNORECASE
    ),
    re.compile(
        r"no\s+issues?\s+identified",
        flags=re.IGNORECASE
    ),
    re.compile(
        r"no\s+violations?\s+(?:were\s+)?found",
        flags=re.IGNORECASE
    ),
    re.compile(
        r"no\s+problem",
        flags=re.IGNORECASE
    )
]


def collect_numeric_matches(
    text,
    patterns
):

    values = []

    for pattern in patterns:

        for match in pattern.finditer(
            text
        ):

            try:
                values.append(
                    int(
                        match.group(1)
                    )
                )
            except Exception:
                pass

    return values


def explicit_no_issue(
    text
):

    return any(
        pattern.search(
            text
        )
        is not None
        for pattern
        in NO_ISSUE_PATTERNS
    )


## 13. Construir evidencia por documento y heurística

Regla conservadora del candidato binario:

- `1`: existe al menos una severidad explícita > 0;
- `0`: no hay severidad > 0 y el texto dice explícitamente que no se encontraron problemas;
- `NaN`: evidencia ambigua o insuficiente.

No se usa `H* Compare` para construir estas etiquetas.


In [17]:
heuristic_evidence_records = []

for _, doc_row in doc_registry.iterrows():

    file_rel = doc_row[
        "file"
    ]

    path = VT_DIR / file_rel

    text = doc_text_cache.get(
        str(path),
        ""
    )

    for heuristic_code, heading in (
        HEURISTICS.items()
    ):

        section = extract_heuristic_section(
            text,
            heading
        )

        if not section:

            heuristic_evidence_records.append({
                "file":
                    file_rel,

                "evaluation_round":
                    doc_row[
                        "evaluation_round"
                    ],

                "prototype":
                    doc_row[
                        "prototype"
                    ],

                "match_status":
                    doc_row[
                        "match_status"
                    ],

                "heuristic":
                    heuristic_code,

                "section_found":
                    False,

                "severity_values":
                    None,

                "overall_rating_values":
                    None,

                "explicit_no_issue":
                    False,

                "label_candidate":
                    np.nan,

                "label_basis":
                    "SECTION_NOT_FOUND"
            })

            continue

        severity_values = collect_numeric_matches(
            section,
            SEVERITY_PATTERNS
        )

        overall_values = collect_numeric_matches(
            section,
            OVERALL_PATTERNS
        )

        no_issue = explicit_no_issue(
            section
        )

        positive_severity = [
            value
            for value
            in severity_values
            if value > 0
        ]

        if positive_severity:

            label_candidate = 1
            label_basis = "EXPLICIT_POSITIVE_SEVERITY"

        elif no_issue:

            label_candidate = 0
            label_basis = "EXPLICIT_NO_ISSUE"

        else:

            label_candidate = np.nan
            label_basis = "AMBIGUOUS_TEXT"

        heuristic_evidence_records.append({
            "file":
                file_rel,

            "evaluation_round":
                doc_row[
                    "evaluation_round"
                ],

            "prototype":
                doc_row[
                    "prototype"
                ],

            "match_status":
                doc_row[
                    "match_status"
                ],

            "heuristic":
                heuristic_code,

            "section_found":
                True,

            "severity_values":
                str(
                    severity_values
                ),

            "overall_rating_values":
                str(
                    overall_values
                ),

            "explicit_no_issue":
                no_issue,

            "label_candidate":
                label_candidate,

            "label_basis":
                label_basis
        })

heuristic_evidence = pd.DataFrame(
    heuristic_evidence_records
)

print(
    "Evidence rows:",
    heuristic_evidence.shape
)

print(
    "\nSection found:"
)

print(
    heuristic_evidence[
        "section_found"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nLabel basis:"
)

print(
    heuristic_evidence[
        "label_basis"
    ].value_counts(
        dropna=False
    )
)

display(
    heuristic_evidence.head(
        100
    )
)


Evidence rows: (172, 11)

Section found:
section_found
True     96
False    76
Name: count, dtype: int64

Label basis:
label_basis
AMBIGUOUS_TEXT       96
SECTION_NOT_FOUND    76
Name: count, dtype: int64


,file,evaluation_round,prototype,match_status,heuristic,section_found,severity_values,overall_rating_values,explicit_no_issue,label_candidate,label_basis
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,1,Bakery,MATCHED_BY_URL,H1,True,[],[],False,NaN,AMBIGUOUS_TEXT
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,1,Bakery,MATCHED_BY_URL,H4,True,[],[],False,NaN,AMBIGUOUS_TEXT
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,1,Bakery,MATCHED_BY_URL,H5,True,[],[],False,NaN,AMBIGUOUS_TEXT
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,1,Bakery,MATCHED_BY_URL,H6,True,[],[],False,NaN,AMBIGUOUS_TEXT
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H1,True,[],[],False,NaN,AMBIGUOUS_TEXT
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H4,True,[],[],False,NaN,AMBIGUOUS_TEXT
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H5,True,[],[],False,NaN,AMBIGUOUS_TEXT
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H6,True,[],[],False,NaN,AMBIGUOUS_TEXT
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,1,NaN,NO_URL_MATCH,H1,True,[],[],False,NaN,AMBIGUOUS_TEXT
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,1,NaN,NO_URL_MATCH,H4,True,[],[],False,NaN,AMBIGUOUS_TEXT


## 14. Resumen de candidatos LLM_DIRECT enlazados por URL

In [18]:
matched_label_evidence = (
    heuristic_evidence[
        heuristic_evidence[
            "match_status"
        ]
        ==
        "MATCHED_BY_URL"
    ]
    .copy()
)

print(
    "Filas de evidencia con prototipo enlazado:",
    len(
        matched_label_evidence
    )
)

print(
    "Prototipos enlazados:",
    matched_label_evidence[
        "prototype"
    ].nunique()
)

print(
    "\nDistribución de candidatos:"
)

print(
    matched_label_evidence[
        "label_candidate"
    ].value_counts(
        dropna=False
    )
)


Filas de evidencia con prototipo enlazado: 64
Prototipos enlazados: 16

Distribución de candidatos:
label_candidate
NaN    64
Name: count, dtype: int64


## 15. Consolidación provisional por prototipo y heurística

Solo se consolida cuando existen etiquetas candidatas no nulas. El resultado conserva número de evaluaciones y acuerdo; todavía se denomina `candidate`, no Y final.


In [19]:
def consolidate_candidate_labels(
    group
):

    valid = (
        group[
            "label_candidate"
        ]
        .dropna()
        .astype(int)
        .tolist()
    )

    if not valid:

        return pd.Series({
            "n_valid_evaluations":
                0,

            "n_positive":
                0,

            "n_negative":
                0,

            "agreement_ratio":
                np.nan,

            "consensus_candidate":
                np.nan,

            "consensus_status":
                "NO_VALID_LABEL"
        })

    n_positive = sum(
        value == 1
        for value
        in valid
    )

    n_negative = sum(
        value == 0
        for value
        in valid
    )

    majority = max(
        n_positive,
        n_negative
    )

    agreement_ratio = (
        majority
        /
        len(valid)
    )

    if n_positive > n_negative:

        consensus = 1
        status = "MAJORITY_POSITIVE"

    elif n_negative > n_positive:

        consensus = 0
        status = "MAJORITY_NEGATIVE"

    else:

        consensus = np.nan
        status = "TIE"

    return pd.Series({
        "n_valid_evaluations":
            len(valid),

        "n_positive":
            n_positive,

        "n_negative":
            n_negative,

        "agreement_ratio":
            agreement_ratio,

        "consensus_candidate":
            consensus,

        "consensus_status":
            status
    })


candidate_consensus_long = (
    matched_label_evidence
    .groupby(
        [
            "prototype",
            "heuristic"
        ],
        group_keys=False
    )
    .apply(
        consolidate_candidate_labels,
        include_groups=False
    )
    .reset_index()
)

display(
    candidate_consensus_long
)


,prototype,heuristic,n_valid_evaluations,n_positive,n_negative,agreement_ratio,consensus_candidate,consensus_status
0,E-Tutor,H1,0,0,0,NaN,NaN,NO_VALID_LABEL
1,E-Tutor,H4,0,0,0,NaN,NaN,NO_VALID_LABEL
2,E-Tutor,H5,0,0,0,NaN,NaN,NO_VALID_LABEL
3,E-Tutor,H6,0,0,0,NaN,NaN,NO_VALID_LABEL
4,Bakery,H1,0,0,0,NaN,NaN,NO_VALID_LABEL
5,Bakery,H4,0,0,0,NaN,NaN,NO_VALID_LABEL
6,Bakery,H5,0,0,0,NaN,NaN,NO_VALID_LABEL
7,Bakery,H6,0,0,0,NaN,NaN,NO_VALID_LABEL
8,Blog app,H1,0,0,0,NaN,NaN,NO_VALID_LABEL
9,Blog app,H4,0,0,0,NaN,NaN,NO_VALID_LABEL


In [20]:
candidate_wide = (
    candidate_consensus_long
    .pivot(
        index="prototype",
        columns="heuristic",
        values="consensus_candidate"
    )
    .reset_index()
)

candidate_wide.columns.name = None

candidate_wide = candidate_wide.rename(
    columns={
        "H1":
            "H1_candidate",

        "H4":
            "H4_candidate",

        "H5":
            "H5_candidate",

        "H6":
            "H6_candidate"
    }
)

print(
    "Candidate wide:",
    candidate_wide.shape
)

display(
    candidate_wide
)


Candidate wide: (16, 5)


,prototype,H1_candidate,H4_candidate,H5_candidate,H6_candidate
0,E-Tutor,NaN,NaN,NaN,NaN
1,Bakery,NaN,NaN,NaN,NaN
2,Blog app,NaN,NaN,NaN,NaN
3,Fintech Mobile App,NaN,NaN,NaN,NaN
4,Food Delivery,NaN,NaN,NaN,NaN
5,Healthcare App,NaN,NaN,NaN,NaN
6,Hospital 1,NaN,NaN,NaN,NaN
7,Hospital 2,NaN,NaN,NaN,NaN
8,KwikTan Membership,NaN,NaN,NaN,NaN
9,Online Shoping,NaN,NaN,NaN,NaN


## 16. Revisar específicamente los prototipos educativos

In [21]:
education_names = set(
    metadata.loc[
        metadata[
            "Domain"
        ]
        .astype(str)
        .str.contains(
            "education",
            case=False,
            na=False
        ),
        "Prototype"
    ]
)

education_label_evidence = (
    matched_label_evidence[
        matched_label_evidence[
            "prototype"
        ].isin(
            education_names
        )
    ]
    .copy()
)

print(
    "Prototipos educativos metadata:",
    sorted(
        education_names
    )
)

print(
    "Educativos enlazados en DOCX:",
    sorted(
        education_label_evidence[
            "prototype"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)

display(
    education_label_evidence
)


Prototipos educativos metadata: [' E-Tutor', 'Quizy', 'True Art']
Educativos enlazados en DOCX: [' E-Tutor', 'Quizy', 'True Art']


,file,evaluation_round,prototype,match_status,heuristic,section_found,severity_values,overall_rating_values,explicit_no_issue,label_candidate,label_basis
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H1,True,[],[],False,NaN,AMBIGUOUS_TEXT
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H4,True,[],[],False,NaN,AMBIGUOUS_TEXT
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H5,True,[],[],False,NaN,AMBIGUOUS_TEXT
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,E-Tutor,MATCHED_BY_URL,H6,True,[],[],False,NaN,AMBIGUOUS_TEXT
40,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,1,True Art,MATCHED_BY_URL,H1,True,[],[],False,NaN,AMBIGUOUS_TEXT
41,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,1,True Art,MATCHED_BY_URL,H4,True,[],[],False,NaN,AMBIGUOUS_TEXT
42,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,1,True Art,MATCHED_BY_URL,H5,True,[],[],False,NaN,AMBIGUOUS_TEXT
43,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,1,True Art,MATCHED_BY_URL,H6,True,[],[],False,NaN,AMBIGUOUS_TEXT
72,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Quizzy.docx,1,Quizy,MATCHED_BY_URL,H1,True,[],[],False,NaN,AMBIGUOUS_TEXT
73,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Quizzy.docx,1,Quizy,MATCHED_BY_URL,H4,True,[],[],False,NaN,AMBIGUOUS_TEXT


## 17. Decisión metodológica provisional

In [22]:
label_semantics_decision = pd.DataFrame([
    {
        "dataset":
            "29042675",

        "compare_as_target":
            False,

        "overall_rating_as_binary_target":
            False,

        "docx_evidence_candidate":
            True,

        "label_provenance":
            "LLM_DIRECT_CANDIDATE",

        "canonical_web_X_ready":
            False,

        "primary_training_ready":
            False,

        "reason":
            "H* Compare contiene filas binarias y filas resumen; no se interpreta como violación. "
            "Los DOCX contienen evaluación heurística explícita y permiten construir candidatos LLM_DIRECT, "
            "pero la fuente visual Figma/Behance/SharePoint no aporta X web canónica equivalente a la extensión."
    }
])

display(
    label_semantics_decision
)


,dataset,compare_as_target,overall_rating_as_binary_target,docx_evidence_candidate,label_provenance,canonical_web_X_ready,primary_training_ready,reason
0,29042675,False,False,True,LLM_DIRECT_CANDIDATE,False,False,"H* Compare contiene filas binarias y filas resumen; no se interpreta como violación. Los DOCX contienen evaluación heurística explícita y permiten construir candidatos LLM_DIRECT, pero la fuente visual Figma/Behance/SharePoint no aporta X web canónica equivalente a la extensión."


## 18. Guardar resultados

In [23]:
metadata_source_audit.to_csv(
    REPORTS_DIR
    / "vt_source_modality_corrected.csv",
    index=False
)

rtf_evidence.to_csv(
    REPORTS_DIR
    / "vt_rtf_semantic_evidence.csv",
    index=False
)

doc_registry.to_csv(
    REPORTS_DIR
    / "vt_docx_evaluation_registry.csv",
    index=False
)

heuristic_evidence.to_csv(
    REPORTS_DIR
    / "vt_docx_heuristic_evidence.csv",
    index=False
)

matched_label_evidence.to_csv(
    INTERIM_DIR
    / "vt_llm_direct_label_candidate_long.csv",
    index=False
)

candidate_consensus_long.to_csv(
    INTERIM_DIR
    / "vt_llm_direct_candidate_consensus_long.csv",
    index=False
)

candidate_wide.to_csv(
    INTERIM_DIR
    / "vt_llm_direct_candidate_wide.csv",
    index=False
)

education_label_evidence.to_csv(
    REPORTS_DIR
    / "vt_education_label_evidence.csv",
    index=False
)

label_semantics_decision.to_csv(
    REPORTS_DIR
    / "vt_label_semantics_decision.csv",
    index=False
)


In [24]:
expected = [
    REPORTS_DIR
    / "vt_source_modality_corrected.csv",

    REPORTS_DIR
    / "vt_rtf_semantic_evidence.csv",

    REPORTS_DIR
    / "vt_docx_evaluation_registry.csv",

    REPORTS_DIR
    / "vt_docx_heuristic_evidence.csv",

    INTERIM_DIR
    / "vt_llm_direct_label_candidate_long.csv",

    INTERIM_DIR
    / "vt_llm_direct_candidate_consensus_long.csv",

    INTERIM_DIR
    / "vt_llm_direct_candidate_wide.csv",

    REPORTS_DIR
    / "vt_education_label_evidence.csv",

    REPORTS_DIR
    / "vt_label_semantics_decision.csv"
]

for path in expected:

    print(
        path.name,
        "✅"
        if path.exists()
        else "❌"
    )


vt_source_modality_corrected.csv ✅
vt_rtf_semantic_evidence.csv ✅
vt_docx_evaluation_registry.csv ✅
vt_docx_heuristic_evidence.csv ✅
vt_llm_direct_label_candidate_long.csv ✅
vt_llm_direct_candidate_consensus_long.csv ✅
vt_llm_direct_candidate_wide.csv ✅
vt_education_label_evidence.csv ✅
vt_label_semantics_decision.csv ✅


## Punto de detención

No renombrar todavía `H1_candidate` → `H1_violation`, etc.

El siguiente paso depende de cuatro resultados:

1. evidencia encontrada en `README.rtf` y `EvaluationTemplate.rtf`;
2. cobertura `MATCHED_BY_URL` de los DOCX;
3. proporción de `EXPLICIT_POSITIVE_SEVERITY`, `EXPLICIT_NO_ISSUE` y `AMBIGUOUS_TEXT`;
4. acuerdo entre evaluaciones por prototipo y heurística.

Después de revisar esos resultados se decidirá si `29042675` entra como `LLM_DIRECT` de referencia o si debe permanecer únicamente como `LABEL_REFERENCE`.
